# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [ ]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [ ]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")

rows = []
row = {}

with open(DATA_PATH,newline="") as f:
    reader = csv.DictReader(f)
    for i in reader:
        rows.append(i)
    print(len(rows))
    print(reader.fieldnames)
    row = rows[0]



5
['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
{'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
   -  ``sample_id`` identifica la muestra.
2. ¿Cuántas características numéricas hay?
   - Las 11 características son numéricas.  
3. ¿Qué columna extra añadirías para probar el contrato?
   - Una columna al final, por ejemplo `extra`.
4. ¿Qué valor de `ph` sería claramente inválido?
   - `ph=2.1` por ejemplo.

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [ ]:
FEATURE_NAMES = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']

class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")
     
    fixed_acidity: float = Field(...,ge=0,le=20, description="Fixed acidity of the wine sample")
    volatile_acidity: float = Field(...,ge=0, le=2, description="Volatile acidity of the wine sample")
    citric_acid: float = Field(...,ge=0, le=2, description="Citric acid of the wine sample")
    residual_sugar: float = Field(...,ge=0,le=20, description="Residual sugar of the wine sample")
    chlorides: float = Field(...,ge=0, le=1, description="Chlorides of the wine sample")
    free_sulfur_dioxide: float = Field(..., ge=0, le=100, description="Free sulfur dioxide of the wine sample")
    total_sulfur_dioxide: float = Field(..., ge=0, le=300, description="Total sulfur dioxide of the wine sample")
    density: float = Field(..., ge=0.98, le=1.01, description="Density of the wine sample")
    ph: float = Field(..., ge=2.5, le=4.5, description="pH of the wine sample")
    sulphates: float = Field(..., ge=0, le=3, description="Sulphates of the wine sample")
    alcohol: float = Field(..., ge=5, le=20, description="Alcohol content of the wine sample")

model_row={}
# TODO: separa sample_id de la fila y valida solo las características del modelo.
for name in FEATURE_NAMES:
    model_row[name] = row[name]
request = WineQualityRequestPydantic.model_validate(model_row)
print(request)

fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [ ]:

invalid_row = {
    "fixed_acidity": 1,
    "volatile_acidity": 0.70,
    "citric_acid": 0.3,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 33.51,
    "sulphates": 0.56,
    "alcohol": 9.4,
}
    
unknown_column_row = {
    "fixed_acidity": 3,
    "volatile_acidity": 0.70,
    "citric_acid": 0.3,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4,
    "unexpected_field": 3
}
# TODO: completa los casos y captura ValidationError.
# try:
#     ...
# except ValidationError as error:
#     print(error)
try:
    request = WineQualityRequestPydantic.model_validate(model_row)
except ValidationError as error:
    print(error)
try:
    request_extra = WineQualityRequestPydantic.model_validate(unknown_column_row)
except ValidationError as error:
    print(error)
try:
    request_invalid = WineQualityRequestPydantic.model_validate(invalid_row)
except ValidationError as error:
    print(error)


1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=33.51, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float
    
instance_row = {
    "fixed_acidity": 3,
    "volatile_acidity": 0.70,
    "citric_acid": 0.3,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 99,
    "sulphates": 0.56,
    "alcohol": 9.4,
}

request = WineQualityRequestDataclass(**instance_row)
# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.

# No pasa nada, ya que no hay ninguún contrato que limite los valores que pueda tomar, simplemente debe ser un Float.

### Decisión de diseño

Completa la frase:

> Usaría Pydantic en valores que vienen del exterior porque deben de cumplir una serie de requisitos que van más allá del tipo (rango de valores para una variable, longitud máxima, mínima, etc.). Usaría `dataclass` en instancias que quiero crear dentro de mi código porque sé cómo las creo y no son ajenas a mí .

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [ ]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']


# TODO: construye vector usando getattr(request, name).
vector = []
for name in FEATURE_NAMES:
    vector.append(getattr(request, name))
    
print(vector)

[3, 0.7, 0.3, 1.9, 0.076, 11.0, 34.0, 0.9978, 99, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- - Sí, claro.
- ¿El clasificador detectaría necesariamente el error?
- El clasificador no, pero el contrato sí. Me daría una predicción errónea
- ¿Cómo evitarías que cada script definiera un orden distinto?
- Definiendo la variable `FEATURE_NAMES` como constante en un módulo accesible para los scripts.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
from typing import Literal

 
WineQualityLabel = Literal["acceptable", "needs_review"]

class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    model_config = ConfigDict(extra="forbidden")
    
    sample_id: str = Field(min_length=1)
    wine_quality: WineQualityLabel = Field()
    confidence: float = Field(ge=0, le=1)
    preprocess_version: str = Field(min_length=1)
    model_version: str = Field(min_length=1)
 

# TODO: escribe un ejemplo de salida para red-001.
prediction_clean = {
    "sample_id": "red-001",
    "wine_quality":"acceptable",
    "confidence": 0.9,
    "preprocess_version": "preprocess-v1",
    "model_version": "model-v1"
}

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Plan de implementación
- `contracts.py`: Aquí declararé los contratos, tanto el de entrada como el de salida.
- `preprocess.py`: Declaro ``PREPROCESS_VERSION``, `MODEL_VERSION`, `FEATURE_NAMES` y la función que dado un objeto de tipo COntratoDeEntrada, devuelva un vector con los valores de cada atributo de dicho objeto coincidente con cada elemento de `FEATURE_NAMES` .
- `inference.py`: carga del artefacto, validación de la entrada, predicción y validación de la salida.
- `predict_file.py`: Leer los datos externos, preprocesarlos, validarlos, cargar el modelo, comparar versiones, comparar features, comprobar labels de predicción, predecir, comprobar salida y exportar.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.